Use a capable model to extract answers and output them in the extraction LLM format
Run only a single JSON file
Constrain the QA LLM output to the specified format

In [ ]:
import json
import os
from PIL import Image
import ast
import textwrap
import base64
import time
from openai import OpenAI
import io
import matplotlib.patches as mpatches

In [41]:
def generate_prompt_q1(sample):
    question_comp = sample["question_comp"]
    items = sample["items"]
    caption_map = sample["caption"]
    
    # caption
    lines = []
    for item in items:
        caption_text = caption_map.get(item, "")
        lines.append(f"the caption of {item}: {caption_text}")
    caption = "\n".join(lines)
    
    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given the following items: {items}.
Each item has a corresponding caption:
{caption}

Question:
{question_comp}

Instructions:
Analyze the provided items to answer the question.
"""

    return prompt

In [42]:
def generate_prompt_q2(q1_resp, sample):
    items = sample["items"]
    caption_map = sample["caption"]
    question_gen = sample["question_gen"]
    question_comp = sample["question_comp"]

    # Output format rules
    format_rule = "```python\n<code>\n```"

    # caption
    lines = []
    for item in items:
        caption_text = caption_map.get(item, "")
        lines.append(f"the caption of {item}: {caption_text}")
    caption = "\n".join(lines)

    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given the following items: {items}.
Each item has a corresponding caption:
{caption}

Previous Question:
{question_comp}

Previous Answer:
{q1_resp}

Question:
{question_gen}

Instructions:
1. Analyze the provided items and the Q1 answer to answer the Q2 question.
2. Do NOT output reasoning, explanation, or intermediate steps.
3. Output ONLY the final answer.

Output format:
Answer: {format_rule}

Rules:
- Output exactly ONE line.
- The line must start with "Answer:".
- Do not output anything before or after the answer.
"""

    return prompt

In [43]:
# Convert chart format and compress images -> reduce tokens
def encode_image_to_base64(image_path, max_size=768, quality=70):
    img = Image.open(image_path)

    # Scale down proportionally
    img.thumbnail((max_size, max_size))
    
    if img.mode != "RGB":
        img = img.convert("RGB")

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality, optimize=True)

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [ ]:
def call_llm_multiround(subject_path, sample):
    
    # Generate the prompt
    prompt_q1 = generate_prompt_q1(sample)

    content_list_q1 = []
    path_map = sample["path"]
    

    for key in path_map:
        relative_path = path_map[key]
        full_path = os.path.join(subject_path, relative_path)

        if not os.path.exists(full_path):
            print(f"File does not exist: {full_path}")
            continue

        mime_type = "image/jpeg"
        image_base64 = encode_image_to_base64(full_path)

        content_list_q1.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_base64}"
            }
        })
        
    content_list_q2 = content_list_q1.copy()

    # Add text
    content_list_q1.append({
        "type": "text",
        "text": prompt_q1
    })
    
    # Call the model
    client = OpenAI(
        api_key="",
        base_url=""
    )
    
    
    attempt = 0
    base_sleep = 1
    max_sleep= 60
    MAX_RETRY = 50 # Avoid an infinite loop    
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")

        try:
            # Q1
            resp_q1 = client.chat.completions.create(
                model="",
                messages=[{"role": "user", "content": content_list_q1}],
                temperature=0
            )
            q1_resp = resp_q1.choices[0].message.content
            
            
            # Q2
            prompt_q2 = generate_prompt_q2(q1_resp, sample)
            content_list_q2.append({
                "type": "text",
                "text": prompt_q2
            })
            
            resp_q2 = client.chat.completions.create(
                model="",
                messages=[{"role": "user", "content": content_list_q2}],
                temperature=0
            )
            q2_resp = resp_q2.choices[0].message.content

            return q2_resp
        
        except Exception as e:
            msg = str(e).lower()
            print(f"[ERROR MSG] {e}")
            
            if ("401" in msg or "403" in msg or "400" in msg or "invalid" in msg or "not found" in msg):
                raise
            
            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1
    
    

In [ ]:
# Use DeepSeek to extract the answer
def exact_answer_llm(output, answer_type):
    
    # Design the prompt
        # Output rules 
    if answer_type == "numeric":
        output_format = "answer: {number}. Return exactly one number after 'answer:'. Do not include units, symbols, commas, explanations, or extra text."

    elif answer_type == "numeric_list":
        output_format = "answer: [a, b, c]. Return a list of numbers inside square brackets. No text inside brackets. No explanations."

    elif answer_type == "choice":
        output_format = "answer: {A capital letter}. Return exactly one capital letter after 'answer:'. No punctuation. No brackets. No extra text."

    elif answer_type == "choice_list":
        output_format = "answer: [A, B, C]. Return a list of capital letters inside square brackets. No explanations."

    elif answer_type == "table_list":
        output_format = "answer: [[row1_col1, row1_col2, ...], [row2_col1, row2_col2, ...], ...]. Return a nested list representing the table. Only structured list format. "

    elif answer_type == "code":
        output_format = "answer: ```python {code block}```. Return only executable code inside the code block. No explanation outside the code block."

    else:
        output_format = "answer: {text}. Return only the final short text answer after 'answer:'. The answer MUST end with a period '.'. Do not include anything after the period. No explanations or additional commentary."
        
    
    prompt = f"""
You are an answer extraction assistant. Your task is to extract the final answer from a model’s response according to the specified answer type. 
Answer type: {answer_type}
Model response:
{output}
You must extract only the final answer and strictly follow this output format:
{output_format}
Do not include explanations or any additional text.
Return exactly one answer in the required format.
    """
        
    # Call the LLM repeatedly until DeepSeek returns successfully
    client = OpenAI(
        api_key="",
        base_url=""
    )
    
    attempt = 0
    base_sleep=1.0
    max_sleep=60.0
    MAX_RETRY = 50 # Avoid an infinite loop
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")
    
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": prompt}
            ],
                max_tokens=1024,
                temperature=0,
                stream=False
            )

            output_llm = response.choices[0].message.content
            
            if not output_llm:
                raise ValueError("empty response")
                
            return output_llm
 
        except Exception as e:
            msg = str(e).lower()

            if ("401" in msg) or ("403" in msg) or ("400" in msg) or ("invalid" in msg) or ("not found" in msg):
                raise

            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1

In [47]:
# Extract Python code
def extract_last_code_block(output):
    output_llm = exact_answer_llm(output, "code")

    block = re.search(
        r"(?is)answer:\s*```(?:python)?\s*\n(.*?)\n```",
        output_llm
    )

    if block:
        return block.group(1)

    return output

# Check code validity
def code_validity(output):
    # print("code_validity")
    if not isinstance(output, str):
        return False
    
    code = textwrap.dedent(output).strip()
    
    try:
        import matplotlib
        matplotlib.use("Agg")
    except Exception:
        pass
    
    try:
        exec(code, {})
        import matplotlib.pyplot as plt
        plt.close("all")
        
        return True
    except Exception:
        pass
    
    try:
        exec(code, {})
        import matplotlib.pyplot as plt
        plt.close("all")
        return True
    except Exception:
        return False
    
# Extract the chart type drawn by the code
# grouped_bar_chart, line_chart, bar_chart, barh_chart, pie_chart, scatter_chart, stacked_bar_chart, box_chart, radar, heatmap
def get_chart_type(code_str: str):
    """
    Extract chart type from plain strings (without using ast)
    Returns:
    grouped_bar_chart, line_chart, bar_chart, barh_chart,
    pie_chart, scatter_chart, stacked_bar_chart,
    box_chart, radar, heatmap
    """

    code = code_str.lower()

    # ----  radar first ----
    if "projection='polar'" in code or 'projection="polar"' in code \
       or "polar=True" in code or "plt.polar(" in code:
        return "radar"

    # ---- heatmap ----
    if any(k in code for k in [
        "imshow(", "matshow(", "pcolormesh(", "pcolor("
    ]):
        return "heatmap"

    # ----  barh ----
    if "barh(" in code:
        return "barh_chart"

    # ----  bar ----
    if "bar(" in code:
        # stacked
        if "bottom=" in code:
            return "stacked_bar_chart"
        
        bar_calls = re.findall(r'bar\((.*?)\)', code)

        for call in bar_calls:
            x_part = call.split(",")[0].replace(" ", "")

            # common grouped bar offset pattern
            grouped_signals = [
                "+width", "-width",
                "+i*", "-i*",
                "+idx*", "-idx*",
                "+j*", "-j*",
                "+bar_width", "-bar_width",
                "+offset", "-offset",
                "width/2", "width*",
            ]

            if any(sig in x_part for sig in grouped_signals):
                return "grouped_bar_chart"
                
        return "bar_chart" 


    # ---- 5 other types ----
    if "boxplot(" in code:
        return "box_chart"

    if "pie(" in code:
        return "pie_chart"

    if "scatter(" in code:
        return "scatter_chart"

    if "plot(" in code:
        return "line_chart"

    return None


# Use regex to extract all [] data from output and return a 2D list
def extract_data(output):
    pattern = r'\[[^\[\]]*\]'
    matches = re.findall(pattern, output)

    out = []
    for m in matches:
        cleaned = m.replace("null", "None")
        try:
            val = ast.literal_eval(cleaned)

            if (
                isinstance(val, list)
                and len(val) > 0
                and all(isinstance(x, (int, float)) for x in val)
            ):
                out.append(val)

        except Exception:
            continue

    return out if out else None

# Determine whether operations were applied to the data
import re

def extract_operation(output: str) -> str:
    code = output.lower()

    lines = code.split("\n")

    # Analyze assignment statements only
    assign_lines = [line for line in lines if "=" in line and "==" not in line]

    joined = " ".join(assign_lines)

    # log
    if any(k in joined for k in ["log(", "np.log", "math.log"]):
        return "log"

    # mean
    if any(k in joined for k in [".mean(", "np.mean", "average("]):
        return "mean"

    # sort
    if any(k in joined for k in ["sorted(", ".sort(", "sort_values("]):
        return "ascending"

    # division
    if "/" in joined or "divide(" in joined:
        return "division"

    # subtraction
    if " - " in joined or "subtract(" in joined:
        return "subtraction"

    return "none"


In [48]:
# Readability checks below
# Some information that can be extracted by running the code is below
import textwrap

def extract_readability(output: str):
    """
    Run the code and read readability information from the matplotlib figure
    No longer includes the label dimension
    """

    code = textwrap.dedent(output).strip()

    try:
        import matplotlib
        matplotlib.use("Agg")
    except Exception:
        pass

    import matplotlib.pyplot as plt

    try:
        exec(code, {"plt": plt})
    except Exception:
        return None

    fig = plt.gcf()
    axes = fig.get_axes()

    if not axes:
        return None

    ax = axes[0]

    # ---------- Basic text ----------
    title = ax.get_title() or None
    xlabel = ax.get_xlabel() or None
    ylabel = ax.get_ylabel() or None
    legend = True if ax.get_legend() else None

    xticklabels = [t.get_text() for t in ax.get_xticklabels()]
    yticklabels = [t.get_text() for t in ax.get_yticklabels()]

    if all(x == "" for x in xticklabels):
        xticklabels = None
        
    if all(y == "" for y in yticklabels):
        yticklabels = None

    # ---------- style ----------
    style = {
        "color": None,
        "linestyle": None,
        "marker": None,
        "figsize": None,
        "fontsize": None
    }

    # figsize
    if fig.get_size_inches() is not None:
        style["figsize"] = True

    # Line chart properties
    for line in ax.get_lines():
        if line.get_marker() not in (None, "None", ""):
            style["marker"] = True
        if line.get_linestyle() not in (None, "None", ""):
            style["linestyle"] = True
        if line.get_color() is not None:
            style["color"] = True

    # Bar chart colors
    for container in ax.containers:
        for bar in container:
            if isinstance(bar, mpatches.Rectangle):
                if bar.get_facecolor() is not None:
                    style["color"] = True

    # fontsize（if the title font differs from the default value）
    try:
        import matplotlib
        if title and ax.title.get_fontsize() != matplotlib.rcParams["axes.titlesize"]:
            style["fontsize"] = True
    except Exception:
        pass

    return {
        "title": title,
        "xlabel": xlabel,
        "ylabel": ylabel,
        "legend": legend,
        "tick": {
            "xticklabels": xticklabels,
            "yticklabels": yticklabels
        },
        "style": style
    }

In [49]:
def rows_match_score(pred_2d, gt_2d, eps=1e-6):
    """
    Row-wise matching:
    - Allow pred to have more rows than gt
    - For each gt row, find the most similar row in pred
    - Return similarity from 0 to 1
    """

    if not isinstance(pred_2d, list) or not isinstance(gt_2d, list):
        return 0.0

    if len(gt_2d) == 0:
        return 0.0

    def row_equal(r1, r2):
        if len(r1) != len(r2):
            return False
        for a, b in zip(r1, r2):
            if a is None and b is None:
                continue
            if isinstance(a, (int, float)) and isinstance(b, (int, float)):
                if abs(a - b) > eps:
                    return False
            else:
                if a != b:
                    return False
        return True

    matched = 0

    for gt_row in gt_2d:
        for pred_row in pred_2d:
            if row_equal(pred_row, gt_row):
                matched += 1
                break   # Finding one is enough

    return matched / len(gt_2d)

In [50]:
# ========= Edit distance similarity =========
def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def levenshtein(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[m]

def edit_sim(a, b):
    # Both are None: full score
    if a is None and b is None:
        return 1.0
    if a is None or b is None:
        return 0.0
    a2 = normalize_text(a)
    b2 = normalize_text(b)
    if len(a2) == 0 and len(b2) == 0:
        return 1.0
    dist = levenshtein(a2, b2)
    denom = max(len(a2), len(b2), 1)
    sim = 1.0 - dist / denom
    return max(0.0, min(1.0, sim))

In [51]:
# Evaluation code —— code
# 1. Whether the provided code can run
# 2. Used data (row matching), chart type, and operations (ascending order, aggregation, etc.)
# 3. Readability
def evaluate_code(output, gt):
    if output is None:
        return False
    
    score = 0

    output = extract_last_code_block(output)

    # 1. Determine whether the code can run
    if(code_validity(output)):
        score = 0.2
    else:
        return 0
    
    print(f"validity score{score}\n")
    
    # 2. Validity 0.45
    # print("Validity")
    # 2.1 Whether the chart type is correct 0.2
    gt_chart_type = gt.get("chart_type")
    pred_chart_type = get_chart_type(output)
    print(f"gt_type: {gt_chart_type}, pred_type: {pred_chart_type}\n")
    if pred_chart_type is not None and gt_chart_type is not None and pred_chart_type == gt_chart_type:
        print(f"chart type score0.2\n")
        score += 0.2
        
    else:
        return 0
    
    # 2.2 Whether the used data is correct 0.25
    gt_data = gt.get("data")  # 2D list
    pred_data = extract_data(output)  # may capture many list
    print(f"gt_data: {gt_data}, \n pred_data: {pred_data}\n")
    if gt_data and pred_data:
        sim = rows_match_score(pred_data, gt_data)
        if sim == 0:
            return 0
        print(f"data score{0.25 * sim}\n")
        score += 0.25 * sim
    
    # 2.3 Whether operations were applied to the data; deduct 0.1
    gt_op = gt.get("operation")
    pred_op = extract_operation(output)
    # print(f"gt_op: {gt_op}, pred_op: {pred_op}")  
    if gt_op is not None:
        if pred_op != gt_op:
            print("operation penalty0.1\n")
            score -= 0.1
    else:
        # When gt does not require an operation, you can also choose not to score it; here it remains no extra points
        pass
    
    # 3. Readability 0.35
    pred = extract_readability(output)
    # print("--------Readability------")
    # print(pred)
    if pred is None:
        return score
    
    # 3.1  Three text fields：Edit distance similarity ---  0.15
    s = 0.0
    s += 0.05 * edit_sim(gt.get("title"),  pred.get("title"))
    s += 0.05 * edit_sim(gt.get("xlabel"), pred.get("xlabel"))
    s += 0.05 * edit_sim(gt.get("ylabel"), pred.get("ylabel"))
    
    print(f"title and label score{s}\n")
    
    # 3.2 Others 0.05 + 0.1 + 0.05
    # Get values
    gt_legend = gt.get("legend")
    pred_legend = pred.get("legend")

    # -------- legend --------
    if gt_legend == pred_legend:
        s += 0.05
        
    tick_items = [
        ("xticklabels", gt.get("tick", {}).get("xticklabels"), pred.get("tick", {}).get("xticklabels")),
        ("yticklabels", gt.get("tick", {}).get("yticklabels"), pred.get("tick", {}).get("yticklabels")),
    ]

    for _, gt_v, pred_v in tick_items:
        if gt_v == pred_v:
            s += 0.05
        elif gt_v is None and gt_legend is not None:
            s += 0.05
            
    # -------- style items --------
    style_items = [
        ("color", gt.get("style", {}).get("color"), pred.get("style", {}).get("color")),
        ("linestyle", gt.get("style", {}).get("linestyle"), pred.get("style", {}).get("linestyle")),
        ("marker", gt.get("style", {}).get("marker"), pred.get("style", {}).get("marker")),
        ("figsize", gt.get("style", {}).get("figsize"), pred.get("style", {}).get("figsize")),
        ("fontsize", gt.get("style", {}).get("fontsize"), pred.get("style", {}).get("fontsize")),
    ]

    for _, gt_v, pred_v in style_items:
        if gt_v == pred_v:
            s += 0.01
        elif gt_v is None and gt_legend is not None:
            s += 0.01
    
    print(f"Readability score{s}\n")
    
    score = score + s 

    return score

In [ ]:
ROOT_PATH = "und_gen"

SUBJECT = "table_to_chart"
JSON_FILE = "comp_to_gen.json"


def evaluate_dataset(subject_path, full_json_path):

    with open(full_json_path, "r", encoding="utf-8") as f:
        samples_data = json.load(f)

    stats = {
        "perceptual": {"score": 0, "count": 0},
        "reasoning": {"score": 0, "count": 0},
        "table to chart": {"score": 0, "count": 0},
        "chart to table": {"score": 0, "count": 0},
    }

    for sample in samples_data:
        print("-----sample-----")
        print(sample)

        output = call_llm_multiround(subject_path, sample)
        gt = sample["answer"]

        score = evaluate_code(output, gt)
        
        print(f"score = {score}")

        question_type = sample["question_type"]

        if question_type in stats:
            stats[question_type]["count"] += 1
            stats[question_type]["score"] += score

    return stats


# ===== Paths =====
subject_path = os.path.join(ROOT_PATH, SUBJECT)
full_json_path = os.path.join(subject_path, JSON_FILE)

if not os.path.exists(full_json_path):
    raise FileNotFoundError(f"{full_json_path} does not exist")

print(f"\n====== Running {SUBJECT} | {JSON_FILE} ======")

# ===== Run =====
stats = evaluate_dataset(subject_path, full_json_path)


# ===== Output results =====
print("\n====== Result ======")

for k in stats:
    total_score = stats[k]["score"]
    total_count = stats[k]["count"]

    avg = total_score / total_count if total_count else 0

    print(f"{k}: Count={total_count}, Avg={avg:.4f}")